# 📖 StoryDiffusion — one character across a comic

Generate a **consistent character across several scenes**, laid out as a comic (training-free).

**Setup:** Runtime → GPU (A100) · free [HF token](https://huggingface.co/settings/tokens) · accept [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev).

### 1 · Install

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf

### 2 · Sign in

In [ ]:
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"   # avoid transient HF-hub read timeouts on big downloads
import torch
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()

### 3 · Load the model (~2–3 min first run)

In [ ]:
from diffusers import ModularPipeline
pipe = ModularPipeline.from_pretrained("remyxai/story-diffusion-flux-modular", trust_remote_code=True)
pipe.load_components(dtype=torch.bfloat16); pipe.to("cuda")
print("✅ ready:", type(pipe.blocks).__name__)  # StoryDiffusionFluxBlock

### 4 · One character across scenes → a comic
Describe the character once + a scene per panel (add `#Caption` for a caption). Edit + run.

In [ ]:
#@title Generate a comic { display-mode: "form" }
CHARACTER = "a young woman with curly red hair and freckles, green jacket"  #@param {type:"string"}
SCENES = "waking up in a sunlit bedroom #Morning | drinking coffee in a cozy kitchen #Coffee | walking through a city park #Afternoon | reading in a warm cafe at night #Evening"  #@param {type:"string"}
seed = 0  #@param {type:"integer"}
import torch
from IPython.display import display
scenes=[s.strip() for s in SCENES.split("|") if s.strip()]
g=torch.Generator("cuda").manual_seed(int(seed))
out=pipe(character_prompt=CHARACTER, scene_prompts=scenes, comic_layout="grid", comic_cols=2,
         height=1024, width=1024, num_inference_steps=28, generator=g).images
sheet=out[0]; sheet.save("result.png"); print("your comic:"); display(sheet.resize((760,780)))

### Download

In [ ]:
from google.colab import files
files.download("result.png")

---
[`remyxai/story-diffusion-flux-modular`](https://huggingface.co/remyxai/story-diffusion-flux-modular) · training-free · non-commercial.